# Real-time CTR (Click-Through Rate) Monitoring

## Problem Statement

You work on a recommendation system that serves **millions of users**. You want to monitor click-through rate (CTR) of items in **real-time**.

### Context
- **Incoming stream**: Each event is either a **click** or **impression**
- **Events arrive as a stream** (not batch)
- **Metrics are noisy** due to small batches of users or low-traffic items
- **Scale**: Must handle high-volume, real-time processing

### Requirements

Implement a Python class that maintains:

1. **Simple Moving Average CTR** over the last N events
   - CTR = clicks / impressions in the window
   - Only stores last N events (sliding window)

2. **Exponential Moving Average CTR** with decay factor `alpha`
   - Gives more weight to recent events
   - Smooths out noise in the data

3. **Constraints**:
   - **Incremental updates**: Cannot store all historical events (memory-efficient)
   - **Real-time queries**: Return current moving averages at any point
   - **Production-ready**: Handle edge cases, validate inputs

### Key Concepts

- **CTR (Click-Through Rate)**: Ratio of clicks to impressions
- **Simple Moving Average**: Average over a fixed window of recent events
- **Exponential Moving Average**: Weighted average where recent events have exponentially more weight

## Approach & Algorithm Design

### High-Level Strategy

1. **Simple Moving Average (SMA)**:
   - Use a **circular buffer** (deque with maxlen) to store last N events
   - Calculate CTR = sum(clicks) / window_size
   - O(1) space, O(1) update time

2. **Exponential Moving Average (EMA)**:
   - Track EMA of clicks and impressions **separately**
   - Formula: `EMA_new = α × current_value + (1 - α) × EMA_old`
   - CTR = EMA_clicks / EMA_impressions
   - O(1) space, O(1) update time

### Why Track Clicks and Impressions Separately?

**Critical Insight**: We cannot directly apply EMA to CTR ratio because:
- CTR = clicks / impressions is a ratio
- EMA of ratios ≠ EMA(clicks) / EMA(impressions)

**Correct approach**: Track EMA of numerator (clicks) and denominator (impressions) separately, then compute the ratio.

### Data Structures

- `deque(maxlen=N)`: Circular buffer for SMA (automatically removes oldest when full)
- Two float variables: For EMA of clicks and impressions
- Counter: Track total events for debugging/monitoring

### Edge Cases to Handle

1. No events recorded → Return `None`
2. Invalid parameters → Raise `ValueError`
3. Window overflow → Automatically handled by deque
4. Division by zero → Check before division


## Time & Space Complexity Analysis

### Time Complexity

| Operation | Time Complexity | Explanation |
|-----------|-----------------|-------------|
| `__init__()` | O(1) | Simple initialization |
| `record_event()` | O(1) | Append to deque + update EMA (constant time) |
| `get_simple_moving_average()` | O(N) | Sum N elements in window |
| `get_exponential_moving_average()` | O(1) | Simple division |
| `get_total_events()` | O(1) | Return counter |

**Note**: `get_simple_moving_average()` is O(N) where N is window_size. For large windows, we could optimize by maintaining a running sum, but for typical window sizes (100-1000), this is acceptable.

### Space Complexity

- **O(N)**: Where N is the window_size for simple moving average
- **O(1)**: For exponential moving average (only stores 2 floats)
- **Total**: O(N) - dominated by the sliding window

### Optimization Opportunity

If window_size is very large (e.g., 1M+), we could optimize `get_simple_moving_average()` to O(1) by:
- Maintaining a running sum of clicks in the window
- On window overflow, subtract the removed element
- Trade-off: Slightly more complex code for better query performance


## Implementation Details

### Key Implementation Points

1. **Initialization**:
   - Validate `window_size > 0` and `0 < alpha <= 1`
   - Initialize deque with `maxlen` for automatic window management
   - Initialize EMA values to 0 (will be set on first event)

2. **Recording Events**:
   - **SMA**: Append 1 (click) or 0 (impression) to deque
   - **EMA**: 
     - First event: Initialize with actual value
     - Subsequent: Apply EMA formula with decay factor

3. **EMA Formula**:
   ```
   EMA_impressions = α × 1.0 + (1 - α) × EMA_impressions_old
   EMA_clicks = α × (1.0 if click else 0.0) + (1 - α) × EMA_clicks_old
   CTR = EMA_clicks / EMA_impressions
   ```

4. **Query Methods**:
   - Return `None` if no events recorded (edge case handling)
   - Check for division by zero
   - Return computed CTR values

### Production Considerations

- **Type hints**: Full type annotations for maintainability
- **Error handling**: Validate inputs, handle edge cases
- **Documentation**: Comprehensive docstrings
- **Memory efficiency**: Only store necessary data
- **Thread safety**: Not implemented (would need locks for concurrent access)


## Code Implementation


In [2]:
import unittest
from collections import deque
from typing import Optional


class CTRMonitor:
    """
    Production-ready class for real-time CTR monitoring with:
    - Simple moving average CTR over last N events
    - Exponential moving average CTR with decay factor alpha
    - Incremental updates without storing all historical events
    """
    
    def __init__(self, window_size: int, alpha: float = 0.1):
        """
        Initialize CTR Monitor.
        
        Args:
            window_size: Number of recent events for simple moving average
            alpha: Decay factor for exponential moving average (0 < alpha <= 1)
        
        Raises:
            ValueError: If window_size <= 0 or alpha not in (0, 1]
        """
        if window_size <= 0:
            raise ValueError("window_size must be positive")
        if not 0 < alpha <= 1:
            raise ValueError("alpha must be in (0, 1]")
        
        self.window_size = window_size
        self.alpha = alpha
        
        # Simple moving average: circular buffer for last N events
        self.event_window = deque(maxlen=window_size)
        
        # Exponential moving average: track EMA of clicks and impressions
        self.ema_clicks = 0.0
        self.ema_impressions = 0.0
        self.total_events = 0
    
    def record_event(self, is_click: bool) -> None:
        """
        Record an event (click or impression).
        
        Args:
            is_click: True if event is a click, False if impression
        """
        # Update simple moving average window
        self.event_window.append(1 if is_click else 0)
        
        # Update exponential moving average
        # For first event, initialize EMA with actual value
        if self.total_events == 0:
            self.ema_impressions = 1.0
            self.ema_clicks = 1.0 if is_click else 0.0
        else:
            # Update EMA: new_ema = alpha * current + (1 - alpha) * old_ema
            self.ema_impressions = self.alpha * 1.0 + (1 - self.alpha) * self.ema_impressions
            self.ema_clicks = self.alpha * (1.0 if is_click else 0.0) + (1 - self.alpha) * self.ema_clicks
        
        self.total_events += 1
    
    def get_simple_moving_average(self) -> Optional[float]:
        """
        Get simple moving average CTR over last N events.
        
        Returns:
            CTR (clicks/impressions) over last N events, or None if no events recorded
        """
        if not self.event_window:
            return None
        
        clicks = sum(self.event_window)
        impressions = len(self.event_window)
        
        if impressions == 0:
            return None
        
        return clicks / impressions
    
    def get_exponential_moving_average(self) -> Optional[float]:
        """
        Get exponential moving average CTR.
        
        Returns:
            Exponential moving average CTR, or None if no events recorded
        """
        if self.total_events == 0:
            return None
        
        if self.ema_impressions == 0:
            return None
        
        return self.ema_clicks / self.ema_impressions
    
    def get_total_events(self) -> int:
        """Get total number of events recorded."""
        return self.total_events


## Testing Strategy (TDD Approach)

### Test-Driven Development (TDD) Process

1. **Red**: Write failing tests first
2. **Green**: Write minimal code to pass tests
3. **Refactor**: Improve code while keeping tests passing

### Test Coverage

Our test suite covers:

1. **Initialization Tests**:
   - Valid parameters
   - Invalid window_size (0, negative)
   - Invalid alpha (0, >1, negative)

2. **Edge Cases**:
   - No events recorded → Returns `None`
   - Single event (click and impression separately)
   - Window overflow (more events than window_size)

3. **Functionality Tests**:
   - Simple moving average with multiple events
   - Exponential moving average with multiple events
   - EMA decay behavior (recent events weighted more)

4. **Integration Tests**:
   - Mixed operations (record + query)
   - High-volume production scenario (10,000 events)
   - Total events counter accuracy

### Why TDD?

- **Confidence**: Tests verify correctness
- **Documentation**: Tests serve as usage examples
- **Refactoring safety**: Can refactor with confidence
- **Interview**: Shows professional development practices


## Test Suite Execution


In [3]:
# TDD: Test Suite
class TestCTRMonitor(unittest.TestCase):
    
    def setUp(self):
        """Set up test fixtures."""
        pass
    
    def test_initialization_valid_params(self):
        """Test initialization with valid parameters."""
        monitor = CTRMonitor(window_size=10, alpha=0.1)
        self.assertEqual(monitor.window_size, 10)
        self.assertEqual(monitor.alpha, 0.1)
        self.assertEqual(monitor.total_events, 0)
    
    def test_initialization_invalid_window_size(self):
        """Test initialization fails with invalid window_size."""
        with self.assertRaises(ValueError):
            CTRMonitor(window_size=0, alpha=0.1)
        with self.assertRaises(ValueError):
            CTRMonitor(window_size=-1, alpha=0.1)
    
    def test_initialization_invalid_alpha(self):
        """Test initialization fails with invalid alpha."""
        with self.assertRaises(ValueError):
            CTRMonitor(window_size=10, alpha=0)
        with self.assertRaises(ValueError):
            CTRMonitor(window_size=10, alpha=1.1)
        with self.assertRaises(ValueError):
            CTRMonitor(window_size=10, alpha=-0.1)
    
    def test_no_events_returns_none(self):
        """Test that methods return None when no events recorded."""
        monitor = CTRMonitor(window_size=10, alpha=0.1)
        self.assertIsNone(monitor.get_simple_moving_average())
        self.assertIsNone(monitor.get_exponential_moving_average())
    
    def test_simple_moving_average_single_event(self):
        """Test simple moving average with single event."""
        monitor = CTRMonitor(window_size=10, alpha=0.1)
        
        # Single click
        monitor.record_event(is_click=True)
        self.assertEqual(monitor.get_simple_moving_average(), 1.0)
        
        # Single impression
        monitor2 = CTRMonitor(window_size=10, alpha=0.1)
        monitor2.record_event(is_click=False)
        self.assertEqual(monitor2.get_simple_moving_average(), 0.0)
    
    def test_simple_moving_average_multiple_events(self):
        """Test simple moving average with multiple events."""
        monitor = CTRMonitor(window_size=5, alpha=0.1)
        
        # Record: click, click, impression, click, impression
        monitor.record_event(is_click=True)   # 1/1 = 1.0
        monitor.record_event(is_click=True)   # 2/2 = 1.0
        monitor.record_event(is_click=False)  # 2/3 = 0.667
        monitor.record_event(is_click=True)   # 3/4 = 0.75
        monitor.record_event(is_click=False)  # 3/5 = 0.6
        
        ctr = monitor.get_simple_moving_average()
        self.assertAlmostEqual(ctr, 0.6, places=5)
    
    def test_simple_moving_average_window_overflow(self):
        """Test simple moving average when window overflows."""
        monitor = CTRMonitor(window_size=3, alpha=0.1)
        
        # Fill window: click, click, impression (CTR = 2/3)
        monitor.record_event(is_click=True)
        monitor.record_event(is_click=True)
        monitor.record_event(is_click=False)
        self.assertAlmostEqual(monitor.get_simple_moving_average(), 2/3, places=5)
        
        # Add more events: should only keep last 3
        monitor.record_event(is_click=False)  # window: [click, impression, impression]
        monitor.record_event(is_click=False)  # window: [impression, impression, impression]
        self.assertEqual(monitor.get_simple_moving_average(), 0.0)
    
    def test_exponential_moving_average_single_event(self):
        """Test exponential moving average with single event."""
        monitor = CTRMonitor(window_size=10, alpha=0.1)
        
        # Single click
        monitor.record_event(is_click=True)
        self.assertEqual(monitor.get_exponential_moving_average(), 1.0)
        
        # Single impression
        monitor2 = CTRMonitor(window_size=10, alpha=0.1)
        monitor2.record_event(is_click=False)
        self.assertEqual(monitor2.get_exponential_moving_average(), 0.0)
    
    def test_exponential_moving_average_multiple_events(self):
        """Test exponential moving average with multiple events."""
        monitor = CTRMonitor(window_size=10, alpha=0.5)  # High alpha for faster decay
        
        # Record events: click, click, impression, click
        monitor.record_event(is_click=True)   # EMA: clicks=1.0, impressions=1.0, CTR=1.0
        monitor.record_event(is_click=True)   # EMA: clicks=1.0, impressions=1.0, CTR=1.0
        monitor.record_event(is_click=False)  # EMA: clicks=0.5, impressions=1.0, CTR=0.5
        monitor.record_event(is_click=True)   # EMA: clicks=0.75, impressions=1.0, CTR=0.75
        
        ctr = monitor.get_exponential_moving_average()
        self.assertAlmostEqual(ctr, 0.75, places=5)
    
    def test_exponential_moving_average_decay(self):
        """Test that exponential moving average gives more weight to recent events."""
        monitor = CTRMonitor(window_size=10, alpha=0.2)
        
        # Record many clicks first
        for _ in range(10):
            monitor.record_event(is_click=True)
        
        initial_ctr = monitor.get_exponential_moving_average()
        self.assertAlmostEqual(initial_ctr, 1.0, places=3)
        
        # Then record many impressions
        for _ in range(10):
            monitor.record_event(is_click=False)
        
        # CTR should decrease but not go to 0 immediately (due to decay)
        final_ctr = monitor.get_exponential_moving_average()
        self.assertLess(final_ctr, 1.0)
        self.assertGreater(final_ctr, 0.0)
    
    def test_mixed_operations(self):
        """Test mixed operations: recording events and querying averages."""
        monitor = CTRMonitor(window_size=5, alpha=0.1)
        
        # Record some events
        for i in range(10):
            monitor.record_event(is_click=(i % 3 == 0))  # Click every 3rd event
        
        # Both averages should be valid
        sma = monitor.get_simple_moving_average()
        ema = monitor.get_exponential_moving_average()
        
        self.assertIsNotNone(sma)
        self.assertIsNotNone(ema)
        self.assertGreaterEqual(sma, 0.0)
        self.assertLessEqual(sma, 1.0)
        self.assertGreaterEqual(ema, 0.0)
        self.assertLessEqual(ema, 1.0)
    
    def test_total_events_counter(self):
        """Test that total events counter is accurate."""
        monitor = CTRMonitor(window_size=3, alpha=0.1)
        
        self.assertEqual(monitor.get_total_events(), 0)
        
        for i in range(10):
            monitor.record_event(is_click=True)
            self.assertEqual(monitor.get_total_events(), i + 1)
        
        self.assertEqual(monitor.get_total_events(), 10)
    
    def test_production_scenario_high_volume(self):
        """Test production scenario with high volume of events."""
        monitor = CTRMonitor(window_size=1000, alpha=0.05)
        
        # Simulate 10,000 events with 5% CTR
        import random
        random.seed(42)
        clicks = 0
        for _ in range(10000):
            is_click = random.random() < 0.05
            if is_click:
                clicks += 1
            monitor.record_event(is_click=is_click)
        
        # Simple moving average should be close to 0.05
        sma = monitor.get_simple_moving_average()
        self.assertGreater(sma, 0.03)
        self.assertLess(sma, 0.07)
        
        # Exponential moving average should also be reasonable
        ema = monitor.get_exponential_moving_average()
        self.assertGreater(ema, 0.0)
        self.assertLess(ema, 1.0)
        
        self.assertEqual(monitor.get_total_events(), 10000)


# Run tests
if __name__ == '__main__':
    unittest.main(argv=[''], exit=False, verbosity=2)


test_exponential_moving_average_decay (__main__.TestCTRMonitor)
Test that exponential moving average gives more weight to recent events. ... ok
test_exponential_moving_average_multiple_events (__main__.TestCTRMonitor)
Test exponential moving average with multiple events. ... ok
test_exponential_moving_average_single_event (__main__.TestCTRMonitor)
Test exponential moving average with single event. ... ok
test_initialization_invalid_alpha (__main__.TestCTRMonitor)
Test initialization fails with invalid alpha. ... ok
test_initialization_invalid_window_size (__main__.TestCTRMonitor)
Test initialization fails with invalid window_size. ... ok
test_initialization_valid_params (__main__.TestCTRMonitor)
Test initialization with valid parameters. ... ok
test_mixed_operations (__main__.TestCTRMonitor)
Test mixed operations: recording events and querying averages. ... ok
test_no_events_returns_none (__main__.TestCTRMonitor)
Test that methods return None when no events recorded. ... ok
test_produc

## Example Usage & Demonstration


In [4]:
# Example Usage
if __name__ == '__main__':
    # Create a CTR monitor with window size 10 and alpha 0.1
    monitor = CTRMonitor(window_size=10, alpha=0.1)
    
    # Simulate a stream of events
    events = [True, True, False, True, False, False, True, False, True, True, False, False]
    
    print("Recording events and monitoring CTR...")
    print("-" * 60)
    
    for i, is_click in enumerate(events, 1):
        monitor.record_event(is_click)
        sma = monitor.get_simple_moving_average()
        ema = monitor.get_exponential_moving_average()
        event_type = "CLICK" if is_click else "IMPRESSION"
        
        # Format values properly (can't use conditional in format specifier)
        sma_str = f"{sma:.3f}" if sma is not None else "N/A"
        ema_str = f"{ema:.3f}" if ema is not None else "N/A"
        
        print(f"Event {i:2d}: {event_type:10s} | "
              f"SMA: {sma_str:>6} | "
              f"EMA: {ema_str:>6} | "
              f"Total: {monitor.get_total_events()}")
    
    print("-" * 60)
    print(f"\nFinal Statistics:")
    final_sma = monitor.get_simple_moving_average()
    final_ema = monitor.get_exponential_moving_average()
    # Format values properly (can't use conditional in format specifier)
    final_sma_str = f"{final_sma:.3f}" if final_sma is not None else "N/A"
    final_ema_str = f"{final_ema:.3f}" if final_ema is not None else "N/A"
    print(f"  Simple Moving Average CTR: {final_sma_str}")
    print(f"  Exponential Moving Average CTR: {final_ema_str}")
    print(f"  Total Events: {monitor.get_total_events()}")


Recording events and monitoring CTR...
------------------------------------------------------------
Event  1: CLICK      | SMA:  1.000 | EMA:  1.000 | Total: 1
Event  2: CLICK      | SMA:  1.000 | EMA:  1.000 | Total: 2
Event  3: IMPRESSION | SMA:  0.667 | EMA:  0.900 | Total: 3
Event  4: CLICK      | SMA:  0.750 | EMA:  0.910 | Total: 4
Event  5: IMPRESSION | SMA:  0.600 | EMA:  0.819 | Total: 5
Event  6: IMPRESSION | SMA:  0.500 | EMA:  0.737 | Total: 6
Event  7: CLICK      | SMA:  0.571 | EMA:  0.763 | Total: 7
Event  8: IMPRESSION | SMA:  0.500 | EMA:  0.687 | Total: 8
Event  9: CLICK      | SMA:  0.556 | EMA:  0.718 | Total: 9
Event 10: CLICK      | SMA:  0.600 | EMA:  0.747 | Total: 10
Event 11: IMPRESSION | SMA:  0.500 | EMA:  0.672 | Total: 11
Event 12: IMPRESSION | SMA:  0.400 | EMA:  0.605 | Total: 12
------------------------------------------------------------

Final Statistics:
  Simple Moving Average CTR: 0.400
  Exponential Moving Average CTR: 0.605
  Total Events: 12


## Interview Tips & Common Pitfalls

### What Interviewers Look For

1. **Correctness**: Does it work for all cases?
2. **Efficiency**: Time/space complexity analysis
3. **Code Quality**: Clean, readable, well-documented
4. **Edge Cases**: Handle None, empty inputs, invalid params
5. **Testing**: Write comprehensive tests

### Common Mistakes to Avoid

1. ❌ **Storing all events**: Violates memory constraint
   - ✅ Use sliding window (deque with maxlen)

2. ❌ **EMA of CTR directly**: `EMA(CTR)` is incorrect
   - ✅ Track EMA of clicks and impressions separately

3. ❌ **No input validation**: Assume valid inputs
   - ✅ Validate window_size > 0, alpha in (0, 1]

4. ❌ **Division by zero**: Don't check before dividing
   - ✅ Check for empty window or zero impressions

5. ❌ **No edge case handling**: Assume events always exist
   - ✅ Return `None` when no events recorded

### Discussion Points

**Q: How would you optimize for very large window sizes?**
- Maintain running sum of clicks in window
- Update sum when adding/removing events
- Reduces `get_simple_moving_average()` from O(N) to O(1)

**Q: How would you make this thread-safe?**
- Add locks (threading.Lock) around shared state
- Consider using `threading.local()` for per-thread instances
- Or use `queue.Queue` for thread-safe event processing

**Q: How would you extend this for multiple items?**
- Use a dictionary: `{item_id: CTRMonitor}`
- Or create a `MultiItemCTRMonitor` class
- Consider memory implications for millions of items

**Q: How would you handle distributed systems?**
- Use distributed counters (Redis, Cassandra counters)
- Aggregate across shards
- Consider eventual consistency trade-offs

### Real-World Applications

- **Recommendation systems**: Monitor item CTR in real-time
- **Ad serving**: Track ad performance metrics
- **A/B testing**: Compare CTR between variants
- **Anomaly detection**: Detect sudden CTR drops
- **Personalization**: Track user engagement metrics
